# FastAPI Introduction with Data Science Applications
This notebook provides an introduction to FastAPI, a modern web framework for building APIs. This project requires Python 3.11+. 
We will explore its integration with data science tools like Pandas and cover aspects like handling requests and validation.

## From a Python function to a web API

A web API makes a Python function available to another program over HTTP. For example, a dashboard can send a feature vector to a prediction service without opening the notebook that trained the model.

The **client** sends a request. The **server** receives it, runs the relevant operation and sends a response. A browser, a Python script or another application can be the client. Client and server can run on the same computer or on different computers.

```text
Client                         Server
Browser / Python script        Uvicorn → FastAPI → Python function
          HTTP request ──────────────────────►
          ◄──────────────────── HTTP response
```

In this tutorial, you will:

- expose Python functions through GET and POST operations;
- send and receive JSON;
- distinguish path parameters, query parameters and a request body;
- explore an API through Swagger UI and its OpenAPI schema;
- validate inputs and describe outputs with Pydantic;
- return an acknowledgement for slow work, then retrieve its status and result;
- apply these ideas to the [image-generation assignment](../assignment/README.md).

The numbered files in `examples/` introduce the concepts one by one. The additional [practice examples](teaching_demo/README.md) connect them in a small numerical prediction service. The prediction is an arithmetic mean, a transparent stand-in for a model call, and does not require a trained model or provider credentials.

## HTTP, REST and JSON

An HTTP request contains a **method**, a **URL**, **headers** and sometimes a **body**. The response contains a **status code**, headers and often a body.

```http
GET / HTTP/1.1
Host: 127.0.0.1:8000
```

```http
HTTP/1.1 200 OK
Content-Type: application/json

{"Hello": "World"}
```

| Method | Typical purpose |
| --- | --- |
| GET | Retrieve a resource or result |
| POST | Submit data for processing or create a resource |
| PUT | Replace a resource |
| PATCH | Update part of a resource |
| DELETE | Remove a resource |

REST is an architectural style. Useful HTTP conventions include identifying resources with URLs, choosing meaningful methods and returning meaningful status codes. Returning JSON alone does not make an API RESTful. A job with an ID is a useful resource; `/predict` is a convenient computation endpoint.

For `http://127.0.0.1:8000/items/?skip=2&limit=3`:

- `127.0.0.1` is the host: the computer making the request;
- `8000` is the port where the server listens;
- `/items/` is the path;
- `skip=2&limit=3` is the query string.

**JSON is text, not a Python object travelling through the network.** Python values are serialized into JSON and parsed back into suitable values by the receiving program.

| JSON | Common Python equivalent |
| --- | --- |
| object: `{"name": "Ada"}` | `dict` |
| array: `[1, 2, 3]` | `list` |
| string, number | `str`, `int` or `float` |
| `true`, `false` | `True`, `False` |
| `null` | `None` |

JSON requires double quotes for strings and object keys. A DataFrame or NumPy array needs conversion to a suitable data structure. Valid JSON also does not include `NaN` or infinity.

Stateless requests do not mean that a server cannot store a model, database records or job results. Each request still needs enough information to identify the operation and relevant resource.

## Setup
Install necessary packages: FastAPI for API development, Uvicorn as an ASGI server, and Pandas for data manipulation.

#### Install packages by syncing the project

Open the Terminal / bash and run 
`uv sync`

### Install and run the examples

Run these commands from **`1_fast_api_tutorial`**:

```bash
uv sync --locked
```

`uv sync --locked` installs the package versions recorded in `uv.lock`. `uv run` executes a command inside this project's environment. The project selects Python 3.11 through `.python-version`.

A notebook cell beginning with `%%file` writes the following code into a Python file. It does **not** start a web server. Execute the first file-writing cell below, then start its app in a terminal:

```bash
uv run --locked uvicorn examples.1_plain_fast:app --reload --port 8000
```

Open `http://127.0.0.1:8000/` and `http://127.0.0.1:8000/docs`. Keep the terminal running. Closing the browser does not stop the server; Ctrl+C in the server terminal does.

Stop the old server before switching examples. Substitute the intended module in the same Uvicorn command:

| File | Module |
| --- | --- |
| `examples/1_plain_fast.py` | `examples.1_plain_fast:app` |
| `examples/2_plain_fast_main.py` | `examples.2_plain_fast_main:app` |
| `examples/3_fast_main.py` | `examples.3_fast_main:app` |
| `examples/4_fast_main.py` | `examples.4_fast_main:app` |
| `examples/5_fast_main.py` | `examples.5_fast_main:app` |
| `examples/6_fast_main.py` | `examples.6_fast_main:app` |

The later files can also be run with `uv run --locked python examples/2_plain_fast_main.py`, substituting the filename. Their `if __name__ == "__main__"` block starts Uvicorn, without automatically enabling reload.

The additional numerical examples use **port 8011** and the same tutorial environment. Their complete files are in `teaching_demo/`; the relevant start command appears with each example below. These are separate apps, so their routes are not automatically available in `examples/`.

## Basic FastAPI Application
Creating a simple FastAPI application with a single route that returns a JSON response. Either run following code or copy it into a python file called `main.py`

In [ ]:
%%file examples/1_plain_fast.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
async def read_root():
    return {"Hello": "World"}


### How the route works

- `FastAPI()` creates the application object.
- `@app.get("/")` registers the combination **GET + `/`**.
- FastAPI calls the function when a matching request arrives.
- The returned dictionary becomes a JSON response.
- Uvicorn runs the server process that receives HTTP traffic.

The function name is a Python detail; the decorator defines the public path. In `module:app`, the part before the colon identifies the Python module and `app` identifies the application object inside it.

With `--reload`, saving a changed Python file restarts the application. This is useful during development, but a restart also clears data stored only in process memory.

**Try it:** change the greeting, save the file and refresh the browser. Find the request's method, path and status code in the terminal log. Entering a URL in the browser address bar sends GET; it does not send a custom JSON POST body.

## Running the FastAPI Application
To run the FastAPI application, use the following command in your terminal:

```bash
uvicorn filename:app --reload
```

Replace `filename` with the name of your Python file containing the FastAPI app. The `--reload` flag is useful during development as it will automatically reload the server when you make changes to your code.


### ...since we are using uv

If you don't activate the virtual environment before running the cli command, you need prefix execution with `uv run`

```bash
uv run uvicorn filename:app --reload
```

For example, if your file is named `main.py`, the command would be:

```bash
uv run uvicorn main:app --reload
```

Open your browser and navigate to `http://127.0.0.1:8000` (or have a look the ports and running server published by the codespace) to see the running application. You can also access the interactive API documentation at `http://127.0.0.1:8000/docs`.


### Running inside codespace

In case you're running the FastAPI app inside Codespace, you need to make the app accessible on all network interfaces, which is necessary for Codespace environments.

```bash
uvicorn filename:app --host 0.0.0.0 --port 8000 --reload
```

# Running the FastAPI application with a main function

If you want to run the FastAPI application from a Python script, you can define a `main` function that starts the Uvicorn server. This is useful when you want to run the application programmatically.

You can then run the script using `python filename.py`.


```python
...
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
```

In [ ]:
%%file examples/2_plain_fast_main.py
from fastapi import FastAPI
import uvicorn

app = FastAPI()

@app.get("/")
async def read_root():
    return {"Hello": "World"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

## Swagger API Documentation

FastAPI automatically generates an interactive API documentation (Swagger UI) that lets you explore the available endpoints and interact with them directly.

You can access the Swagger UI at the `/docs` endpoint of your FastAPI application. For example, if your application is running locally, you can access it at `http://localhost:8000/docs`.

With Swagger you can see the available endpoints, make requests, and view the responses directly in your browser.

<img src="images/swagger_img.png">

There is also a JSON version of the API documentation available at the `/openapi.json` endpoint.

### Explore an API with Swagger UI

1. Open `/docs` for the app you are running.
2. Expand an operation and select **Try it out**.
3. Enter parameters or a request body, then select **Execute**.
4. Inspect **Request URL**, **Server response**, the status code, **Response body** and **Response headers**.
5. Compare the generated `curl` command with the request you just sent.

The **Server response** shows the actual result of this request. The documented **Responses** section describes possible responses; it is not proof that a particular request succeeded.

| Address | What it provides |
| --- | --- |
| `/docs` | Swagger UI: an interactive API client and documentation page |
| `/openapi.json` | OpenAPI: the machine-readable API description |
| `/redoc` | An alternative documentation view |

FastAPI generates OpenAPI from routes, parameters and data models. Swagger UI uses that description to display and call the API. Other tools can use it to generate clients too.

![Swagger UI showing the complete numerical practice application](images/handout/01-swagger-overview.png)

*This screenshot shows `teaching_demo/step05.py` on port 8011. The first example has only GET `/`; additional routes appear as the application grows.*

**Try it:** open `/openapi.json` and find `paths`. Identify the operation corresponding to the route you executed in Swagger.

## Paths and Parameters

You can define different paths and parameters in FastAPI to create more complex APIs. Here's an example of a path with a parameter:

```python

@app.get("/hello/{name}")
def root(name: str):
    return {"message": f"hello {name}"}
```

In this example, the path `/hello/{name}` contains a parameter `name`. When you make a request to `/hello/world`, the value of `name` will be `"world"`.

### Automatic type validation
FastAPI automatically validates the type of the parameter based on the type annotation. If the type doesn't match, it will return a 422 error with a detailed error message.


In [ ]:
%%file examples/3_fast_main.py

from fastapi import FastAPI
import pandas as pd
import uvicorn

app = FastAPI()

@app.get("/")
async def read_root():
    return {"Hello": "World"}


@app.get("/hello/{name}")
def hello_path(name: str):
    return {"hello" : f"{name}"}


@app.get("/square/{num}")
def get_square(num: int):
    return {"result" : num * num}


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

### Path parameters and validation

In `/square/{num}`, the variable part of the path matches the function parameter `num`. The annotation `num: int` tells FastAPI to parse and validate the incoming value as an integer before calling the function.

| Request to the current example | Result |
| --- | --- |
| `/square/4` | 200 with `{"result":16}` |
| `/square/-4` | 200 with `{"result":16}` |
| `/square/banana` | 422: the value cannot be parsed as an integer |
| `/squarre/4` | 404: the path does not exist |

Ordinary Python type annotations do not generally enforce types when you call a function directly. FastAPI uses them at the HTTP boundary. A negative integer is valid here because no rule forbids it.

Read a validation error's **location** first, then its **message** and **type**. Swagger may reject an invalid integer in its form before sending it. A browser request or `curl` lets you see the server's response directly:

```bash
curl -i http://127.0.0.1:8000/square/banana
```

![An integer path parameter fails validation with HTTP 422](images/handout/02-path-validation.png)

*The equivalent numerical practice example calls its parameter `number`, so this screenshot names `number`. The example above calls it `num`; its error location will name `num`.*

### Query Parameters

You can also define query parameters in FastAPI by adding parameters to the endpoint function with default values.

```python
@app.get("/items/")
def read_item(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}
```

In [ ]:
%%file examples/4_fast_main.py

from fastapi import FastAPI
import uvicorn
import pandas as pd

app = FastAPI()

@app.get("/")
async def read_root():
    return {"Hello": "World"}

@app.get("/hello/{name}")
def hello_path(name: str):
    return {"hello" : f"{name}"}


@app.get("/square/{num}")
def get_square(num: int):
    return {"result" : num * num}


@app.get("/items/")
def read_item(skip: int = 0, limit: int = 10):
    # random df with 100 entries
    # return based on skip and limit
    df = pd.DataFrame({"entries": range(100)})
    return df.iloc[skip:skip+limit].to_dict(orient="records")

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
    


### Query parameters, pagination and Pandas

The path identifies a resource or collection. Query parameters select a view of it. In the example above, the DataFrame contains 100 rows, while `skip` and `limit` select the rows to return.

| Request on port 8000 | Result |
| --- | --- |
| `/items/` | Entries 0–9 |
| `/items/?skip=2&limit=2` | `[{"entries":2},{"entries":3}]` |
| `/items/?skip=100` | An empty array |
| `/items/?skip=banana` | 422 |

`to_dict(orient="records")` converts rows into a list of dictionaries. This gives the client a JSON array of row objects instead of a Python-specific DataFrame.

The current example declares integer types, but no numerical bounds. Negative values therefore follow Python slicing; they do not automatically cause validation errors. To require nonnegative `skip` and a `limit` between 1 and 10, replace that endpoint with:

```python
from fastapi import Query

@app.get("/items/")
def read_item(skip: int = Query(0, ge=0), limit: int = Query(10, ge=1, le=10)):
    df = pd.DataFrame({"entries": range(100)})
    return df.iloc[skip:skip + limit].to_dict(orient="records")
```

**Try it:** compare `skip=-1` and `limit=0` before and after adding the bounds. Check which constraints appear in `/docs`.

The separate numerical practice app uses `/items` without a trailing slash, four records and a default limit of 3. Its results differ because its data and defaults differ.

### Use HTTP Status Codes

You can return different HTTP status codes from your FastAPI endpoints to indicate the success or failure of a request.

```python

from fastapi import HTTPException

@app.get("/items/{item_id}")
def read_item(item_id: int):
    if item_id == 0:
        raise HTTPException(status_code=404, detail="Item not found")
    return {"item_id": item_id}

```

In this example, if the `item_id` is `0`, the endpoint will return a `404 Not Found` error with the message `"Item not found"`.
You should use Status Codes in your API responses to provide meaningful information about the request outcome.


### Status codes are part of the contract

| Code | Meaning in these examples |
| --- | --- |
| 200 | The request was handled successfully |
| 202 | Work was accepted; the result may not exist yet |
| 404 | A route or requested resource was not found |
| 405 | The path exists, but not for this HTTP method |
| 422 | Declared request validation failed |
| 500 | A server-side failure |

Returning `{"error":"Not found"}` normally still produces HTTP 200. Choose the status explicitly, for example by raising `HTTPException(status_code=404, detail="Item not found")`.

An unknown route and an unknown resource can both produce 404. A known route with invalid input produces 422 in these examples. A failing output schema is a server bug, rather than invalid client input.

The `item_id` code block above is a standalone illustration. Avoid registering multiple conflicting `/items/{...}` routes in one app just to combine different examples.

## Respond with binary files, like images

You can also return binary files from FastAPI endpoints, such as images or other media files.

```python
from fastapi.responses import FileResponse

@app.get("/image/")
def get_image():
    return FileResponse("img.png")

```

### Return JSON or image bytes

`Content-Type` tells the client how to interpret a response body. A JSON response commonly uses `application/json`; a PNG image uses `image/png`.

`FileResponse` serves an existing file. In the snippet above, `"img.png"` is resolved relative to the server's working directory. When running from `1_fast_api_tutorial`, the supplied image is at **`images/img.png`**:

```python
from fastapi.responses import FileResponse

@app.get("/image/", response_class=FileResponse)
def get_image():
    return FileResponse("images/img.png", media_type="image/png")
```

This endpoint does not generate an image. Generating an image, storing it and serving it later are separate responsibilities.

If an image generator already gives you bytes, import `Response` with `from fastapi import Response` and use `Response(content=image_bytes, media_type="image/png")` instead of serving a file. First check that generation actually returned valid image data.

### POST Method, Request Parameters and JSON

You can also define POST methods in FastAPI to receive data from the client. You can define request parameters by directly adding them to the endpoint function, or you can use Pydantic models for more complex data structures.

The advantage of using Pydantic models is that FastAPI will automatically validate the request data based on the model schema.

```python

from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd

app = FastAPI()
df = pd.DataFrame(columns=["name", "description", "price", "tax"])

class Item(BaseModel):
    name: str
    description: str | None = None
    price: float
    tax: float | None = None

@app.post("/items/")
async def create_item(item: Item):
    df.loc[len(df)] = pd.Series(item.model_dump())
    return {"item": item}
```

In [ ]:
%%file examples/5_fast_main.py

from fastapi import FastAPI, HTTPException
import uvicorn
import pandas as pd
from pydantic import BaseModel


class Item(BaseModel):
    name: str
    description: str | None = None
    price: float
    tax: float | None = None


app = FastAPI()
## static list to store items
df = pd.DataFrame(columns=["name", "description", "price", "tax"])



## endpoint to add items to the df
@app.post("/items/")
async def create_item(item: Item):
    df.loc[len(df)] = pd.Series(item.model_dump())
    return {"item": item}


##also add an endpoint to get all items and search by name
@app.get("/items/")
def get_items(name: str=None):
    if name:
        return df[df["name"] == name].to_dict(orient="records")
    return df.to_dict(orient="records")


## get items with path params, search by name
@app.get("/items/{name}")
def get_item(name: str):
    filtered_df = df[df["name"] == name]
    if not filtered_df.empty:
        # Convert the first matched row to a dictionary
        return filtered_df.iloc[0].to_dict()
    else:
        raise HTTPException(status_code=404, detail="Item not found")


## python main entry point

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)


### Request bodies and Pydantic models

In POST `/items/`, the `Item` model describes the JSON body. It is a data-validation model, not a machine-learning model.

```json
{"name": "demo", "price": 2.5}
```

`name` and `price` are required. `description` and `tax` may be omitted or set to `null`, because their annotations allow `None` and their defaults are `None`.

FastAPI parses the body, asks Pydantic to validate it and passes an `Item` object to the function. `item.model_dump()` converts that object into a Python dictionary. The DataFrame is an in-memory store for this exercise; restarting the server resets it.

**Try it:** create a complete item and an item with only `name` and `price`. Retrieve them through GET `/items/`, filter by `name`, and request `/items/a-name-that-does-not-exist`. Check the response shapes and status codes.

Types and constraints define the API boundary. They do not guarantee that input is scientifically meaningful: feature order, units and the model's training distribution also matter.

### A numerical prediction API

The next practice example accepts a feature vector and returns its arithmetic mean. Replace the mean calculation with a trained model call when you need real inference; the request and response contract can remain stable.

From `1_fast_api_tutorial`, run the complete example on port 8011:

```bash
uv run --locked uvicorn step03:app --app-dir teaching_demo --reload --port 8011
```

Its central code is:

```python
from statistics import fmean
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI()

class PredictionRequest(BaseModel):
    features: list[float] = Field(min_length=1, max_length=100)

class PredictionResponse(BaseModel):
    prediction: float
    model: str

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    return {"prediction": fmean(request.features), "model": "demo-mean-v1"}
```

`PredictionRequest` requires an object containing a list of 1–100 numeric values. Sending only the array would be a different contract. `PredictionResponse` describes and validates the output and appears in the generated documentation.

POST `/predict` with `{"features":[1,2,3]}` returns 200 with `{"prediction":2.0,"model":"demo-mean-v1"}`. The `model` field is application metadata; the HTTP status is protocol metadata.

![A successful numerical prediction in Swagger UI](images/handout/03-prediction-success.png)

*This is the separate numerical app on port 8011. POST `/predict` is not a route in the item-storage example on port 8000.*

### Explore the request contract

Try these bodies in POST `/predict`:

| Body | HTTP result | Reason |
| --- | --- | --- |
| `{"features":[1,2,3]}` | 200 | Valid input |
| `{}` | 422 | Required field missing |
| `{"features":[]}` | 422 | List too short |
| `{"features":[1,"banana"]}` | 422 | An element cannot be parsed as a float |

![A request-body validation error in Swagger UI](images/handout/05-body-validation.png)

Pydantic can convert compatible values, such as a numeric string, into numbers. Validation does not mean every value with a different original Python/JSON type is rejected. Strict validation can be configured when required.

Open `/predict` directly in the browser address bar. It sends GET to a path that only supports POST and receives **405 Method Not Allowed**. The method and path together identify an operation.

Inspect **Schemas** in `/docs` and the corresponding entries in `/openapi.json`. Find the request's list constraints and the response fields.

### Exercise: add POST `/sum`

Extend the numerical app with POST `/sum`. Reuse `PredictionRequest` and return a JSON object containing `total`, the sum of the features.

| Input | Expected result |
| --- | --- |
| `{"features":[1,2,3]}` | 200 with `{"total":6.0}` |
| `{"features":[-2,2]}` | 200 with `{"total":0.0}` |
| `{"features":[]}` | 422 |

Work with a partner: one person implements the endpoint, and the other discovers and tests it using Swagger. Swap roles after the first successful request.

Extend the exercise by adding a response model with `total: float`. Verify that its schema appears in the documentation. Consider which information a client still needs beyond types, such as feature names, units and ordering.

### Slow requests and asynchronous code

A slow calculation can delay the response without stopping the whole server. The complete next example is `teaching_demo/step04.py`:

```bash
uv run --locked uvicorn step04:app --app-dir teaching_demo --reload --port 8011
```

It adds this endpoint to the numerical app:

```python
import time

@app.post("/predict-slow", response_model=PredictionResponse)
def slow_predict(request: PredictionRequest):
    time.sleep(5)
    return predict(request)
```

`predict(request)` here is a local Python function call, not another HTTP request. The sleep represents a slow operation. This request waits about five seconds for its result.

| Pattern | Behaviour |
| --- | --- |
| Normal `def` route | FastAPI runs it in a thread pool; its client waits until a response is ready |
| `async def` with an asynchronous `await` | Can yield control while waiting; its client still waits for the function's response |
| `async def` containing `time.sleep(...)` | Blocks the event-loop thread during that sleep |
| `BackgroundTasks` | Runs work after the response has been sent |

Changing only `def` to `async def` does not make computation instant or turn a blocking SDK into an asynchronous SDK. A thread pool has finite capacity and is not a promise of parallel CPU-heavy Python execution across cores.

**Try it:** send a request to `/predict-slow`, then request GET `/` in another tab during the wait. Distinguish the response time of one request from whether the server can handle other requests.

## Ready to serve a ml model

So much for the basics, let's move on to more advanced topics. In our case we will use FastAPI to build a simple machine learning model and serve it as an API. 

Slow tasks can delay a response. FastAPI BackgroundTasks runs a task after the response is sent, in the same application process. It does not create a separate worker process or a durable job queue. For CPU-intensive model computation, use a separate worker process or a task queue. Store the task result explicitly so a later request can retrieve it; returning from the task does not send another response to the client.

```python
from fastapi import FastAPI, BackgroundTasks


## endpoint with background task
@app.post("/predict/")
async def predict(item: Item, background_tasks: BackgroundTasks):
    background_tasks.add_task(run_model, item)
    return response # usually a message that the model is running

def run_model(item: Item):
    # run the model
    return prediction
```

### From a background task to a job API

The preceding ML code block is a **design sketch**: names such as `response`, `prediction` and `run_model` are placeholders. The next file-writing cell is a runnable example that writes a prompt to `log.txt`. Its message does not mean an image has actually been generated.

A useful job API needs more than scheduling a function:

1. create an identifier and an initial job record;
2. return an acknowledgement;
3. run the work and store its outcome;
4. let a later request retrieve the current state or result.

Returning a value from a background task does not send a second response to the original request. The response has already been sent, so the result must be stored explicitly.

In [ ]:
%%file examples/6_fast_main.py

from typing import Optional

from fastapi import FastAPI, BackgroundTasks
from fastapi.requests import Request
from pydantic import BaseModel
import uvicorn


class ImageRequest(BaseModel):
    prompt: str


app = FastAPI()

# Function to be run as a background task.
# This is just a placeholder function for demonstration.
# In your application, this could be a function that generates an image.
def write_log(message: str):
    # Example of a time-consuming task: Writing a message to a file.
    # Replace this with the logic of your image generation task.
    with open("log.txt", "a") as file:
        file.write(f"{message}\n")


@app.post("/item")
async def root(image_request: ImageRequest, background_tasks: BackgroundTasks):
    prompt = image_request.prompt
    background_tasks.add_task(write_log, prompt)
    return {"message": "Image generation has started."}


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

## Submit a job and retrieve its result

Run the complete numerical job example from `1_fast_api_tutorial`:

```bash
uv run --locked uvicorn step05:app --app-dir teaching_demo --reload --port 8011
```

The following code extends the numerical prediction app with a temporary job store. The complete file also includes the request models and prediction functions used earlier.

```python
import time
from uuid import uuid4
from fastapi import BackgroundTasks, HTTPException

jobs = {}

@app.post("/jobs", status_code=202)
def create_job(request: PredictionRequest, background_tasks: BackgroundTasks):
    job_id = str(uuid4())
    job = {"job_id": job_id, "status": "queued"}
    jobs[job_id] = job
    background_tasks.add_task(run_prediction_job, job_id, request)
    return job


def run_prediction_job(job_id: str, request: PredictionRequest):
    jobs[job_id] = {"job_id": job_id, "status": "running"}
    try:
        time.sleep(5)
        result = predict(request)
        jobs[job_id] = {
            "job_id": job_id, "status": "succeeded", "result": result
        }
    except Exception:
        jobs[job_id] = {
            "job_id": job_id,
            "status": "failed",
            "error": "Prediction failed. Please try again.",
        }


@app.get("/jobs/{job_id}")
def get_job(job_id: str):
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail="Job not found")
    return jobs[job_id]
```

`BackgroundTasks` is supplied by FastAPI; the client does not include it in the JSON body. `add_task` receives the function itself and its arguments. Calling the function immediately, as in `run_prediction_job(...)`, would run it at that point instead of passing it for later execution.

The job record exists before scheduling the work. `uuid4()` supplies an identifier that connects the submission to later retrieval requests. Define the complete worker before submitting a job.

### Observe the job lifecycle

```text
POST /jobs → 202 with job_id
                 │
                 ▼
              queued → running → succeeded + result
                          └────→ failed + error

GET /jobs/{job_id} → current stored record
```

1. Submit `{"features":[1,2,3]}` to POST `/jobs` in Swagger.
2. Copy the returned `job_id`.
3. Execute GET `/jobs/{job_id}` immediately.
4. Execute it again after the work finishes.

This repeated checking is **polling**. Each retrieval is a separate HTTP request.

![POST /jobs returns a 202 acknowledgement and a job identifier](images/handout/04-job-accepted.png)

202 acknowledges acceptance, not completion. The first GET may already show `running` or `succeeded`; you are not guaranteed to observe every intermediate state.

![A later GET returns a completed job and its prediction result](images/handout/06-job-result.png)

The job state and HTTP status answer different questions. HTTP 200 can mean a job record was retrieved successfully even when that record's state is `failed`. An unknown job ID produces 404.

These states are part of the application design. FastAPI does not create or maintain them automatically.

### Process memory, task queues and reliability

The job dictionary belongs to one server process. Restarting the server, including a save with `--reload`, clears its records. Do not use multiple Uvicorn workers with this in-memory design: requests could reach processes with different dictionaries.

FastAPI `BackgroundTasks` does not provide durable storage, automatic retries, a separate worker service or guaranteed execution after a process failure. Synchronous background functions run in a thread pool within the same application process.

For independent workers and recovery, applications may use persistent storage and a task queue. The assignment's optional **RQ** extension uses Redis/Valkey. **RabbitMQ** is a message broker that you may encounter in other worker architectures; it is not a model or a replacement for FastAPI. A broker alone does not guarantee reliable results: storage, retries and failure handling still need a design.

**Try it:** finish a job, restart the app, then request the old ID. Explain the resulting 404. This illustrates loss of in-memory state, not every possible crash scenario.

## Apply the pattern to the image assignment

The final numerical example also provides GET `/sample-image`. Open `http://127.0.0.1:8011/sample-image` to retrieve a fixed PNG file. It is not the output of a prediction job.

Its file path is based on the Python file's location, so it does not depend on the terminal's working directory:

```python
from pathlib import Path
from fastapi.responses import FileResponse

@app.get("/sample-image", response_class=FileResponse)
def sample_image():
    image_path = Path(__file__).parent / "assets" / "demo.png"
    return FileResponse(image_path, media_type="image/png")
```

The [assignment README](../assignment/README.md) defines the required image routes:

| Numerical practice example | Image assignment |
| --- | --- |
| Feature-vector request | Prompt request and custom prompt structure |
| POST `/jobs` | POST `/images` |
| Job identifier and stored state | Image ID, state and result location |
| GET `/jobs/{job_id}` | GET `/image/{image_id}` |
| Simulated slow calculation | `ImageGenerator.generate_image(...)` |
| JSON prediction result | Generated image bytes or an image file |

Keep **singular** `/image/{image_id}` for retrieval. A separate status endpoint is an optional design choice, not an additional requirement.

One possible response contract is:

| Situation | Example response |
| --- | --- |
| Submission accepted | 202 with an image ID |
| Known image still processing | 202 with a processing message |
| Image ready | 200 with `Content-Type: image/png` and image bytes |
| Unknown image ID | 404 |
| Generation failed | A documented failure status and error body |

These are suggested conventions, not a claim that the assignment specifies every status code. Document the design you choose.

The provided generator is synchronous. A normal `def` background function fits that interface; wrapping a blocking call in `async def` does not make it non-blocking. The wrapper can return `None` if no image artifact is produced. Handle that case and provider exceptions as failed generation, rather than marking the image ready.

### Build the assignment in small steps

1. Accept a valid prompt and return an identifier.
2. Recognize known and unknown identifiers in the retrieval endpoint.
3. Simulate a delay and make a prepared image available afterwards.
4. Connect the provided generator.
5. Expose processing, success and failure outcomes clearly.
6. Document how to start the app and reproduce a submission/retrieval sequence.

Relevant files:

- [assignment/README.md](../assignment/README.md): requirements and deliverables;
- [assignment/app.py](../assignment/app.py): starter app and TODOs;
- [image_generator.py](../assignment/image_generator/image_generator.py): prompt template and generator interface;
- [stability_api.py](../assignment/image_generator/stability_api.py): synchronous provider call and returned image bytes.

The numerical examples do not implement the image assignment. You still design image storage and retrieval, integrate the provider and build your custom prompt structure. Profanity checking and RQ remain optional extensions.

The numerical examples do not need API keys. Provider credentials for the assignment belong in server-side configuration, not in the client's JSON prompt. The private course repository supplies a `.env` at its root; do not paste its contents into requests or screenshots.

## Call the API from another client

Swagger is one client. The same method, URL and body can be used from a terminal or a Python program.

With `step05` running on port 8011, run these commands in another terminal. They use POSIX shell quoting; use suitable quoting in PowerShell or send the requests through Swagger.

```bash
curl -i http://127.0.0.1:8011/
curl -i http://127.0.0.1:8011/square/banana
curl -i 'http://127.0.0.1:8011/items?skip=2&limit=2'
curl -i -X POST http://127.0.0.1:8011/predict \
  -H 'Content-Type: application/json' \
  -d '{"features":[1,2,3]}'
curl -i -X POST http://127.0.0.1:8011/jobs \
  -H 'Content-Type: application/json' \
  -d '{"features":[1,2,3]}'
```

Copy the returned job ID into `http://127.0.0.1:8011/jobs/YOUR_JOB_ID`. Refreshing that page sends a new GET request.

A Python client can send the same prediction request:

```python
import httpx

response = httpx.post(
    "http://127.0.0.1:8011/predict",
    json={"features": [1, 2, 3]},
)
response.raise_for_status()
print(response.json())
```

### Codespaces

From `1_fast_api_tutorial`, start the numerical app with:

```bash
uv run --locked uvicorn step05:app --app-dir teaching_demo --host 0.0.0.0 --port 8011 --reload
```

Open the forwarded port in Codespaces and append `/docs` to that URL. In a local browser, `127.0.0.1` means your own computer, not the remote Codespace. If you use a different port, update the client URL as well.

## Troubleshooting

| Symptom or question | What to check |
| --- | --- |
| Connection refused | Is Uvicorn running? Did startup fail? Does the client use the correct host and port? |
| Could not import a module | Run from `1_fast_api_tutorial`. Match the module and app name; include `--app-dir teaching_demo` for the numerical examples. |
| Address already in use | Stop the earlier server with Ctrl+C, or choose another port and update the URL. |
| New endpoint missing in Swagger | Save the file being served, wait for reload and refresh `/docs`. Each checkpoint is an independent app. |
| Swagger is blank | Its UI assets normally load from a CDN. Check `/openapi.json` and try `curl`; the API can work while the UI is unavailable. |
| A numeric string was accepted | Pydantic can parse compatible values. Use explicit strictness if your contract requires it. |
| 422 versus 500 | 422 indicates declared input validation failed in these examples; 500 indicates a server-side failure. |
| An old job ID now returns 404 | A restart or reload cleared the in-memory store. Submit a new job. |
| TestClient waits for the background job | TestClient waits for the application call and its background tasks. Use real HTTP requests against Uvicorn to observe an early acknowledgement. |
| A task returned a value but the client did not receive it | Store the result for a later request; the original response was already sent. |
| A provider returned no image | Check for missing bytes and exceptions; record a visible failure state. |

The local numerical app has no authentication. Public deployment introduces additional authentication, rate limits and operational requirements; it is not needed to practise local requests.

Check the supplied examples from `1_fast_api_tutorial`:

```bash
uv run --locked python -m unittest discover -s ../tests -v
uv run --locked python teaching_demo/smoke_check.py
```

The first command checks the original examples and SDK compatibility without calling providers. The second also starts a temporary local server to check image serving and the difference between a slow response and a prompt job acknowledgement.

If you need more practice with environments, use the [uv setup notebook](../0_uv_tutorial/uv_setup_tutorial.ipynb). Packaging and custom CLI entrypoints are separate topics; `uv sync` and `uv run` are sufficient for this tutorial.

## Check your understanding

- Trace one request from client to Python function and back to a response.
- Explain the difference between Swagger UI and OpenAPI.
- Give one example each of a path parameter, query parameter and request body.
- Explain why valid JSON does not necessarily satisfy a Pydantic model.
- Explain why schema-valid features may still be unsuitable for an ML model.
- Explain why `async def` alone does not produce an immediate response.
- Explain why 202 can arrive before the result exists, and where the result must be stored.
- Describe the responses for a processing image, a ready image, an unknown ID and failed generation.

## Further reading

- [MDN: HTTP overview](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Overview)
- [MDN: Working with JSON](https://developer.mozilla.org/en-US/docs/Learn_web_development/Core/Scripting/JSON)
- [FastAPI: First steps](https://fastapi.tiangolo.com/tutorial/first-steps/)
- [FastAPI: Request bodies](https://fastapi.tiangolo.com/tutorial/body/)
- [Pydantic: Models and validation](https://docs.pydantic.dev/latest/concepts/models/)
- [FastAPI: Response models](https://fastapi.tiangolo.com/tutorial/response-model/)
- [FastAPI: Concurrency and async/await](https://fastapi.tiangolo.com/async/)
- [Starlette: Thread pools](https://starlette.dev/threadpool/)
- [FastAPI: Background tasks](https://fastapi.tiangolo.com/tutorial/background-tasks/)
- [RQ: Job queues](https://python-rq.org/)
- [RabbitMQ: Producers and consumers](https://www.rabbitmq.com/tutorials/tutorial-one-python)

The numerical examples share this tutorial's environment and lockfile. The screenshots show the supplied numerical app; the numbered examples may expose different routes or parameter names as described alongside each screenshot.